# Query-plan LLM lab

Build query-plan requests from scratch, then test task batch size, parallel request count, provider route, and reasoning effort.

Supported provider modes:
- `openrouter`: OpenRouter chat completions, optionally routed to Cerebras with `provider.order`.
- `cerebras`: direct Cerebras OpenAI-compatible chat completions endpoint.

Live calls are disabled by default. Set `RUN_LIVE_CALLS = True` in the config cell when you intentionally want to spend API quota.

In [1]:
from __future__ import annotations

import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from pprint import pprint
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

repo_root = Path.cwd()
if not (repo_root / "backend" / "app").exists():
    repo_root = repo_root.parent
assert (repo_root / "backend" / "app").exists(), f"Could not locate repo root from {Path.cwd()}"
sys.path.insert(0, str(repo_root / "backend"))

os.environ.setdefault("POSTGRES_HOST", "localhost")
os.environ.setdefault("POSTGRES_PORT", "5432")
os.environ.setdefault("POSTGRES_DB", "coverage_dashboard")
os.environ.setdefault("POSTGRES_USER", "postgres")
os.environ.setdefault("SOURCE_DATA_ROOT", str(repo_root / "Datasets"))

from app.datasets import build_random_sample_bundle_from_db
from app.experiment_runner import (
    QUERY_PLAN_SYSTEM_PROMPT,
    QUERY_PLAN_USER_TEMPLATE,
    _extract_json_object,
    _normalize_plan,
)

def env_first(*names: str) -> str:
    for name in names:
        value = os.environ.get(name, "").strip()
        if value:
            return value
    return ""

## Experiment knobs

In [9]:
RUN_LIVE_CALLS = True

# Fresh sample source. These are the same fields the experiment runner uses for random sampling.
SAMPLE_CONFIG = {
    "requested_datasets": ["2t-red"],
    "dataset_sample_size": 1,
    "tables_per_dataset": 1,
    "records_per_table": 80,
    "random_seed": 42,
    "context_rows": 2,
}

# Provider mode: "openrouter" or "cerebras".
PROVIDER_MODE = "openrouter"

# OpenRouter can route to Cerebras without changing the endpoint.
OPENROUTER_ROUTE_PROVIDER = "cerebras"
OPENROUTER_ALLOW_FALLBACKS = True

# Direct Cerebras uses CEREBRAS_API_KEY and https://api.cerebras.ai/v1/chat/completions.
MODELS = {
    "openrouter": os.environ.get("OPENROUTER_MODEL", "openai/gpt-oss-120b"),
    "cerebras": os.environ.get("CEREBRAS_MODEL", "gpt-oss-120b"),
}

TASK_LIMITS = [8, 16, 32, 64]
REASONING_EFFORTS = ["low", "medium", "high"]
PARALLEL_REQUESTS = [2, 4, 8, 16]

TEMPERATURE = 0.1
MAX_TOKENS = 4096
TIMEOUT_SECONDS = 180
MAX_RETRIES_PER_CASE = 5

## Build fresh query-plan samples

In [10]:
bundle = build_random_sample_bundle_from_db(SAMPLE_CONFIG)
samples = bundle["samples"]

print(f"samples={len(samples)} warnings={len(bundle.get('warnings') or [])}")
pprint(bundle.get("sampling_manifest")[:5])
pprint(samples[:2])

samples=80 warnings=0
[{'available_records': 213,
  'dataset': '2t-red',
  'sampled_records': 80,
  'table_id': 'Q7CDPWKD'}]
[{'col_id': 0,
  'dataset': '2t-red',
  'dataset_dir': '2t-red',
  'gt_qids': ['Q228', 'Q12916633', 'Q19746247'],
  'gt_raw_value': 'http://www.wikidata.org/entity/Q228 '
                  'http://www.wikidata.org/entity/Q12916633 '
                  'http://www.wikidata.org/entity/Q19746247',
  'gt_source_file': 'cea_gt.csv',
  'header': ['col0', 'col1', 'col2', 'col3', 'col4', 'col5', 'col6'],
  'header_cell': 'col0',
  'lookup_context': ['Co-Prince Emmanuel Macron',
                     '14 May 2017',
                     '2 years, 204 days',
                     'Flusp rur ompeurav da',
                     'Ex officcio',
                     'Principality of Andorra',
                     'Co-Prince Archbishop Joan Enric Vives Sicíliaa',
                     '12 May 2003',
                     '16 years, 206 days',
                     'Fluemp groomp',
     

## Request builder

This mirrors the production query-plan prompt, but lets the notebook vary provider-specific parameters.

In [11]:
def task_payloads(batch_samples: list[dict]) -> list[dict]:
    return [
        {
            "id": sample["sample_id"],
            "mention": sample.get("mention_text"),
            "lookup_text": sample.get("lookup_text"),
            "dataset": sample.get("dataset"),
            "table_id": sample.get("table_id"),
            "row_id": sample.get("row_id"),
            "col_id": sample.get("col_id"),
            "context_for_reasoning_only": {
                "header": sample.get("header"),
                "target_row": sample.get("target_row"),
                "rows_before": sample.get("rows_before"),
                "rows_after": sample.get("rows_after"),
                "lookup_context": sample.get("lookup_context"),
            },
        }
        for sample in batch_samples
    ]


def build_query_plan_body(
    batch_samples: list[dict],
    *,
    provider_mode: str,
    model: str,
    reasoning_effort: str,
    max_tokens: int | None,
    temperature: float,
) -> dict:
    user_payload = {
        **QUERY_PLAN_USER_TEMPLATE,
        "tasks": task_payloads(batch_samples),
        "output_schema": {
            "tasks": [
                {
                    "id": "same id",
                    "optimized_query": "short query",
                    "normalized_mention": "normalized mention or null",
                    "typo_corrected_mention": "high-confidence correction or null",
                    "typo_correction_confidence": 0.0,
                    "typo_correction_reason": "short reason or null",
                    "coarse_type": "PERSON | ORGANIZATION | LOCATION | EVENT | WORK | PRODUCT | CONCEPT | MISC | null",
                    "fine_type": "PERSON | FICTIONAL_CHARACTER | COMPANY | NONPROFIT_ORG | GOVERNMENT_ORG | EDUCATIONAL_ORG | SPORTS_TEAM | COUNTRY | CITY | REGION | US_STATE | LANDMARK | CELESTIAL_BODY | CONFLICT | SPORT_EVENT | EVENT_GENERIC | FILM | BOOK | MUSIC_WORK | SOFTWARE | INTERNET_MEME | DEVICE | MEDICATION | FOOD_BEVERAGE | PRODUCT_GENERIC | LANGUAGE | LAW | SCIENTIFIC_THEORY | BIOLOGICAL_TAXON | ANATOMY | MISC | null",
                    "wikipedia_url": "page slug or null",
                    "dbpedia_url": "resource slug or null",
                    "aliases": ["short alternatives"],
                    "context_expansion_terms": ["short discriminating terms"],
                }
            ]
        },
    }
    body = {
        "model": model,
        "temperature": temperature,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": QUERY_PLAN_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
        ],
    }
    if max_tokens is not None:
        body["max_tokens"] = int(max_tokens)

    if provider_mode == "openrouter":
        body["include_reasoning"] = False
        body["reasoning"] = {"effort": reasoning_effort}
        if OPENROUTER_ROUTE_PROVIDER:
            body["provider"] = {
                "order": [OPENROUTER_ROUTE_PROVIDER],
                "allow_fallbacks": bool(OPENROUTER_ALLOW_FALLBACKS),
            }
    elif provider_mode == "cerebras":
        body["reasoning_effort"] = reasoning_effort
    else:
        raise ValueError(f"Unsupported provider_mode={provider_mode!r}")
    return body


preview_body = build_query_plan_body(
    samples[:2],
    provider_mode=PROVIDER_MODE,
    model=MODELS[PROVIDER_MODE],
    reasoning_effort=REASONING_EFFORTS[0],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
)
print(json.dumps({k: v for k, v in preview_body.items() if k != "messages"}, indent=2))
print("prompt_chars=", len(json.dumps(preview_body, ensure_ascii=False)))

{
  "model": "openai/gpt-oss-120b",
  "temperature": 0.1,
  "response_format": {
    "type": "json_object"
  },
  "max_tokens": 4096,
  "include_reasoning": false,
  "reasoning": {
    "effort": "low"
  },
  "provider": {
    "order": [
      "cerebras"
    ],
    "allow_fallbacks": true
  }
}
prompt_chars= 5338


## Direct provider caller

In [12]:
def provider_settings(provider_mode: str) -> dict:
    if provider_mode == "openrouter":
        return {
            "endpoint": os.environ.get("OPENROUTER_CHAT_URL", "https://openrouter.ai/api/v1/chat/completions"),
            "api_key": env_first("OPENROUTER_API_KEY", "LLM_API_KEY"),
            "model": MODELS["openrouter"],
            "headers": {
                "HTTP-Referer": os.environ.get("OPENROUTER_SITE_URL", "http://localhost"),
                "X-Title": os.environ.get("OPENROUTER_APP_NAME", "query-plan-llm-lab"),
            },
        }
    if provider_mode == "cerebras":
        return {
            "endpoint": os.environ.get("CEREBRAS_CHAT_URL", "https://api.cerebras.ai/v1/chat/completions"),
            "api_key": env_first("CEREBRAS_API_KEY"),
            "model": MODELS["cerebras"],
            "headers": {},
        }
    raise ValueError(f"Unsupported provider_mode={provider_mode!r}")


def message_content_to_text(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)


def parse_query_plan_response(content_text: str, batch_samples: list[dict]) -> dict:
    parsed = _extract_json_object(content_text)
    raw_tasks = parsed.get("tasks") if isinstance(parsed.get("tasks"), list) else []
    by_id = {sample["sample_id"]: sample for sample in batch_samples}
    plans = {}
    unknown_ids = []
    invalid_count = 0
    for raw_task in raw_tasks:
        if not isinstance(raw_task, dict) or not raw_task.get("id"):
            invalid_count += 1
            continue
        task_id = str(raw_task["id"])
        sample = by_id.get(task_id)
        if not sample:
            unknown_ids.append(task_id)
            continue
        plans[task_id] = _normalize_plan(raw_task, sample)
    requested_ids = list(by_id)
    missing_ids = [task_id for task_id in requested_ids if task_id not in plans]
    return {
        "parsed": parsed,
        "raw_tasks": raw_tasks,
        "plans": plans,
        "requested_ids": requested_ids,
        "returned_ids": [str(task.get("id")) for task in raw_tasks if isinstance(task, dict) and task.get("id") is not None],
        "unknown_ids": unknown_ids,
        "invalid_count": invalid_count,
        "missing_ids": missing_ids,
    }


def call_query_plan_batch(
    batch_samples: list[dict],
    *,
    provider_mode: str,
    reasoning_effort: str,
    max_tokens: int | None = MAX_TOKENS,
    temperature: float = TEMPERATURE,
    timeout_seconds: int = TIMEOUT_SECONDS,
    max_retries: int = MAX_RETRIES_PER_CASE,
) -> dict:
    settings = provider_settings(provider_mode)
    if not settings["api_key"]:
        raise RuntimeError(f"Missing API key for {provider_mode}. Set OPENROUTER_API_KEY or CEREBRAS_API_KEY.")
    body = build_query_plan_body(
        batch_samples,
        provider_mode=provider_mode,
        model=settings["model"],
        reasoning_effort=reasoning_effort,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    attempts = []
    final_payload = {}
    final_parse = {}
    last_error = None
    started_all = time.monotonic()

    for attempt_index in range(1, max(1, int(max_retries)) + 1):
        started = time.monotonic()
        headers = {
            "Authorization": f"Bearer {settings['api_key']}",
            "Content-Type": "application/json",
            **settings.get("headers", {}),
        }
        request = Request(
            settings["endpoint"],
            data=json.dumps(body).encode("utf-8"),
            headers=headers,
            method="POST",
        )
        try:
            with urlopen(request, timeout=timeout_seconds) as response:
                raw_response = (response.read() or b"{}").decode("utf-8", errors="replace")
                payload = json.loads(raw_response)
                message = (payload.get("choices") or [{}])[0].get("message") or {}
                content_text = message_content_to_text(message.get("content"))
                parsed = parse_query_plan_response(content_text, batch_samples)
                final_payload = payload
                final_parse = parsed
                attempts.append(
                    {
                        "attempt": attempt_index,
                        "http_status": getattr(response, "status", None),
                        "duration_seconds": round(time.monotonic() - started, 3),
                        "content_present": bool(content_text.strip()),
                        "usable": len(parsed["plans"]),
                        "missing": len(parsed["missing_ids"]),
                        "error": None,
                    }
                )
                if parsed["missing_ids"] and attempt_index < max_retries:
                    time.sleep(min(30.0, 4.0 * attempt_index))
                    continue
                break
        except HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            last_error = f"HTTP {exc.code}: {detail[:500]}"
            attempts.append({"attempt": attempt_index, "http_status": exc.code, "duration_seconds": round(time.monotonic() - started, 3), "error": last_error})
            if attempt_index < max_retries and (exc.code in {408, 409, 425, 429} or exc.code >= 500):
                time.sleep(min(60.0, 4.0 * attempt_index))
                continue
            break
        except (URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = str(exc)
            attempts.append({"attempt": attempt_index, "duration_seconds": round(time.monotonic() - started, 3), "error": last_error})
            if attempt_index < max_retries:
                time.sleep(min(30.0, 4.0 * attempt_index))
                continue
            break

    usage = final_payload.get("usage") or {}
    completion_details = usage.get("completion_tokens_details") or {}
    return {
        "provider_mode": provider_mode,
        "model": settings["model"],
        "reasoning_effort": reasoning_effort,
        "requested": len(batch_samples),
        "usable": len(final_parse.get("plans") or {}),
        "missing": len(final_parse.get("missing_ids") or []),
        "returned": len(final_parse.get("returned_ids") or []),
        "unknown": len(final_parse.get("unknown_ids") or []),
        "invalid": final_parse.get("invalid_count", 0),
        "attempts": attempts,
        "request_chars": len(json.dumps(body, ensure_ascii=False)),
        "duration_seconds": round(time.monotonic() - started_all, 3),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "reasoning_tokens": completion_details.get("reasoning_tokens"),
        "response_id": final_payload.get("id"),
        "error": last_error,
    }

## Single-call sweep: tasks x reasoning effort

Set `PROVIDER_MODE` to `openrouter` or `cerebras` in the config cell, then run this with `RUN_LIVE_CALLS = True`.

In [13]:
def run_task_reasoning_sweep(provider_mode: str) -> list[dict]:
    results = []
    for task_limit in TASK_LIMITS:
        batch_samples = samples[:task_limit]
        for effort in REASONING_EFFORTS:
            result = call_query_plan_batch(
                batch_samples,
                provider_mode=provider_mode,
                reasoning_effort=effort,
                max_retries=MAX_RETRIES_PER_CASE,
            )
            results.append(result)
            pprint(result)
    return results


if RUN_LIVE_CALLS:
    task_reasoning_results = run_task_reasoning_sweep(PROVIDER_MODE)
else:
    print("Live calls disabled. Set RUN_LIVE_CALLS = True to run the sweep.")

{'attempts': [{'attempt': 1,
               'content_present': True,
               'duration_seconds': 1.896,
               'error': None,
               'http_status': 200,
               'missing': 0,
               'usable': 8}],
 'completion_tokens': 1622,
 'duration_seconds': 1.896,
 'error': None,
 'invalid': 0,
 'missing': 0,
 'model': 'openai/gpt-oss-120b',
 'prompt_tokens': 4846,
 'provider_mode': 'openrouter',
 'reasoning_effort': 'low',
 'reasoning_tokens': 371,
 'request_chars': 16861,
 'requested': 8,
 'response_id': 'gen-1782914718-DxL4HP3zJsODsOJqT9jC',
 'returned': 8,
 'unknown': 0,
 'usable': 8}
{'attempts': [{'attempt': 1,
               'content_present': True,
               'duration_seconds': 7.096,
               'error': None,
               'http_status': 200,
               'missing': 0,
               'usable': 8}],
 'completion_tokens': 3537,
 'duration_seconds': 7.096,
 'error': None,
 'invalid': 0,
 'missing': 0,
 'model': 'openai/gpt-oss-120b',
 'prompt

## Parallel request sweep

This checks client-side parallelism for a fixed `tasks_per_request` and reasoning effort. Keep retries at 1 while finding provider limits.

In [14]:
def chunks(items: list[dict], size: int) -> list[list[dict]]:
    return [items[index : index + size] for index in range(0, len(items), size)]


def run_parallel_sweep(provider_mode: str, *, tasks_per_request: int = 4, reasoning_effort: str = "low") -> list[dict]:
    batches = chunks(samples, tasks_per_request)[: max(PARALLEL_REQUESTS)]
    all_results = []
    for parallelism in PARALLEL_REQUESTS:
        selected = batches[:parallelism]
        started = time.monotonic()
        with ThreadPoolExecutor(max_workers=parallelism) as executor:
            futures = [
                executor.submit(
                    call_query_plan_batch,
                    batch,
                    provider_mode=provider_mode,
                    reasoning_effort=reasoning_effort,
                    max_retries=MAX_RETRIES_PER_CASE,
                )
                for batch in selected
            ]
            results = [future.result() for future in as_completed(futures)]
        aggregate = {
            "provider_mode": provider_mode,
            "parallelism": parallelism,
            "tasks_per_request": tasks_per_request,
            "reasoning_effort": reasoning_effort,
            "requests": len(results),
            "requested_tasks": sum(item["requested"] for item in results),
            "usable_tasks": sum(item["usable"] for item in results),
            "missing_tasks": sum(item["missing"] for item in results),
            "wall_seconds": round(time.monotonic() - started, 3),
            "child_results": results,
        }
        all_results.append(aggregate)
        pprint(aggregate)
    return all_results


if RUN_LIVE_CALLS:
    parallel_results = run_parallel_sweep(PROVIDER_MODE, tasks_per_request=4, reasoning_effort="low")
else:
    print("Live calls disabled. Set RUN_LIVE_CALLS = True to run the parallel sweep.")

{'child_results': [{'attempts': [{'attempt': 1,
                                  'content_present': True,
                                  'duration_seconds': 1.306,
                                  'error': None,
                                  'http_status': 200,
                                  'missing': 0,
                                  'usable': 4}],
                    'completion_tokens': 1005,
                    'duration_seconds': 1.306,
                    'error': None,
                    'invalid': 0,
                    'missing': 0,
                    'model': 'openai/gpt-oss-120b',
                    'prompt_tokens': 2866,
                    'provider_mode': 'openrouter',
                    'reasoning_effort': 'low',
                    'reasoning_tokens': 346,
                    'request_chars': 10242,
                    'requested': 4,
                    'response_id': 'gen-1782915559-SWHSe3wWONH6fAfwHc7E',
                    'returned': 4,
        

## Compare OpenRouter-routed Cerebras vs direct Cerebras

This runs the same batch against both provider modes. You need both `OPENROUTER_API_KEY` and `CEREBRAS_API_KEY` set.

In [15]:
COMPARE_TASKS = 4
COMPARE_REASONING_EFFORT = "low"

if RUN_LIVE_CALLS:
    comparison_results = []
    for provider_mode in ["openrouter", "cerebras"]:
        comparison_results.append(
            call_query_plan_batch(
                samples[:COMPARE_TASKS],
                provider_mode=provider_mode,
                reasoning_effort=COMPARE_REASONING_EFFORT,
                max_retries=MAX_RETRIES_PER_CASE,
            )
        )
    pprint(comparison_results)
else:
    print("Live calls disabled. Set RUN_LIVE_CALLS = True to compare providers.")

[{'attempts': [{'attempt': 1,
                'content_present': True,
                'duration_seconds': 1.455,
                'error': None,
                'http_status': 200,
                'missing': 0,
                'usable': 4}],
  'completion_tokens': 980,
  'duration_seconds': 1.455,
  'error': None,
  'invalid': 0,
  'missing': 0,
  'model': 'openai/gpt-oss-120b',
  'prompt_tokens': 2533,
  'provider_mode': 'openrouter',
  'reasoning_effort': 'low',
  'reasoning_tokens': 278,
  'request_chars': 8991,
  'requested': 4,
  'response_id': 'gen-1782915583-R95jI4NAei65w8rsQzQI',
  'returned': 4,
  'unknown': 0,
  'usable': 4},
 {'attempts': [{'attempt': 1,
                'duration_seconds': 0.221,
                'error': 'HTTP 403: error code: 1010\n',
                'http_status': 403}],
  'completion_tokens': None,
  'duration_seconds': 0.221,
  'error': 'HTTP 403: error code: 1010\n',
  'invalid': 0,
  'missing': 0,
  'model': 'gpt-oss-120b',
  'prompt_tokens': None,
  '

## Optional historical failure sanity check

Use this only to compare live experiments against the stored `webapp_experiment_job_3` failure pattern.

In [17]:
from app.db import connect

with connect() as conn:
    rows = list(
        conn.execute(
            """
            SELECT
                id,
                task_count,
                status,
                error,
                jsonb_array_length(coalesce(response_metadata->'attempts', '[]'::jsonb)) AS requests,
                response_metadata->'usage' AS usage,
                response_metadata->>'response_content' AS response_content
            FROM llm_prompt_batches
            WHERE run_id = (SELECT id FROM runs WHERE name = 'webapp_experiment_job_3')
              AND status = 'failed'
            ORDER BY id
            """
        ).fetchall()
    )

for row in rows:
    usage = row.get("usage") or {}
    details = usage.get("completion_tokens_details") or {}
    print(
        f"batch={row['id']} requests={row['requests']} content={row['response_content']!r} "
        f"completion_tokens={usage.get('completion_tokens')} reasoning_tokens={details.get('reasoning_tokens')}"
    )

batch=58 requests=5 content='None' completion_tokens=40000 reasoning_tokens=39997
batch=157 requests=5 content='None' completion_tokens=40000 reasoning_tokens=39997
batch=169 requests=5 content='None' completion_tokens=40000 reasoning_tokens=39997
batch=170 requests=5 content='None' completion_tokens=40000 reasoning_tokens=39997
